In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-5 : GET RAW INVALID RECORDS FROM QUARANTINE
# =========================================================

invalid_df = spark.read.table("retails.silver.customers_quarantine") \
                    .filter((col("_rescued_data").isNotNull()) & (col("quarantine_status") == 'NEW'))

In [0]:
from pyspark.sql.functions import col, get_json_object as get_json_from

# =========================================================
# STEP-6 : TRY TO RECOVER RESCUED COLUMNS
# =========================================================
recovered_df = invalid_df \
                .withColumn("customer_id_fixed", get_json_from(col("_rescued_data"), "$.customer_id")) \
                .withColumn("customer_fname_fixed", get_json_from(col("_rescued_data"), "$.customer_fname")) \
                .withColumn("customer_lname_fixed", get_json_from(col("_rescued_data"), "$.customer_lname")) \
                .withColumn("customer_email_fixed", get_json_from(col("_rescued_data"), "$.customer_email")) \
                .withColumn("customer_password_fixed", get_json_from(col("_rescued_data"), "$.customer_password")) \
                .withColumn("customer_street_fixed", get_json_from(col("_rescued_data"), "$.customer_street")) \
                .withColumn("customer_city_fixed", get_json_from(col("_rescued_data"), "$.customer_city")) \
                .withColumn("customer_state_fixed", get_json_from(col("_rescued_data"), "$.customer_state")) \
                .withColumn("customer_zipcode_fixed", get_json_from(col("_rescued_data"), "$.customer_zipcode"))


In [0]:
from pyspark.sql.functions import coalesce, trim

# =========================================================
# STEP-7 : MERGE RECOVERED VALUES
# =========================================================

recovered_df = recovered_df \
    .withColumn(
        "customer_id",
        coalesce(col("customer_id"), col("customer_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "customer_fname",
        coalesce(col("customer_fname"), trim(col("customer_fname_fixed")))
    ) \
    .withColumn(
        "customer_lname",
        coalesce(col("customer_lname"), col("customer_lname_fixed"))
    ) \
    .withColumn(
        "customer_email",
        coalesce(col("customer_email"), col("customer_email_fixed"))
    ) \
    .withColumn(
        "customer_password",
        coalesce(col("customer_password"), col("customer_password_fixed"))
    ) \
    .withColumn(
        "customer_street",
        coalesce(col("customer_street"), col("customer_street_fixed"))
    ) \
    .withColumn(
        "customer_city",
        coalesce(col("customer_city"), col("customer_city_fixed"))
    ) \
    .withColumn(
        "customer_state",
        coalesce(col("customer_state"), col("customer_state_fixed"))
    ) \
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode"), col("customer_zipcode_fixed").cast("integer"))
    )

In [0]:
recovered_df = recovered_df.drop("customer_id_fixed", "customer_fname_fixed", "customer_lname_fixed", "customer_email_fixed", "customer_password_fixed", "customer_street_fixed", "customer_city_fixed", "customer_state_fixed", "customer_zipcode_fixed")

In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-8 : APPLY DATA QUALITY RULES
# =========================================================

cleaned_recovered_df = recovered_df.filter(
    col("customer_id").isNotNull() &
    col("customer_fname").isNotNull() &
    col("customer_lname").isNotNull() &
    col("customer_email").isNotNull() &
    col("customer_zipcode").isNotNull() &
    (col("customer_zipcode") >= 0)
)

In [0]:
cleaned_recovered_df = cleaned_recovered_df.dropDuplicates(["customer_id"])
cleaned_recovered_df.createOrReplaceTempView("customers_cleaned_vw_fixed")

In [0]:
from pyspark.sql.functions import when, current_timestamp, sha2, concat_ws

cleaned_recovered_df = cleaned_recovered_df \
    .withColumn("customer_id", col("customer_id").cast("bigint")) \
    .withColumn("customer_zipcode", col("customer_zipcode").cast("bigint")) \
    .withColumn("batch_id", col("batch_id").cast("integer")) \
    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False)) 

cleaned_recovered_df = cleaned_recovered_df.select("customer_id", "customer_fname", "customer_lname", "customer_email", "customer_password", "customer_street", "customer_city", "customer_state", "customer_zipcode", "op", "is_deleted", "source_system", "source_file_name", "ingestion_ts", "ingestion_dt", "batch_id", "run_id")

cleaned_recovered_df = cleaned_recovered_df.withColumn("event_ts", current_timestamp()) \
                    .withColumn("record_hash",
                                sha2(
                                    concat_ws(
                                        "||",
                                        col("customer_id"),
                                        col("customer_fname"),
                                        col("customer_lname"),
                                        col("customer_email"),
                                        col("customer_street"),
                                        col("customer_city"),
                                        col("customer_state"),
                                        col("customer_zipcode")
                                    ),
                                    256
                                )
                            )
    

try:
    cleaned_recovered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("retails.silver.customers_cdc")

except Exception as e:
    print(str(e))

In [0]:
merge_query_rescued = """
    MERGE INTO retails.silver.customers_quarantine t
    USING customers_cleaned_vw_fixed s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN
        UPDATE SET t.quarantine_status = 'FIXED', t.reprocessed_at = current_timestamp()
 
    """
spark.sql(merge_query_rescued).show()


In [0]:
update_query_corrupt = """
        UPDATE retails.silver.customers_quarantine
        SET
            quarantine_status = 'INVALID',
            reprocessed_at = current_timestamp()
        WHERE quarantine_status = 'NEW'
"""

spark.sql(update_query_corrupt).show()

In [0]:
%sql
-- select * from retails.silver.customers_quarantine;
-- select * from retails.silver.customers_cleaned;

-- describe retails.silver.customers_quarantine;